# 01 - Extração: SQL Server → MinIO (CSV)

Extrai **todas as tabelas** do database `SeguroDB` do SQL Server e exporta como CSV para o bucket `landing-zone` no MinIO.

**Pré-requisitos:** Docker Compose rodando, Notebook `00` executado.

## 1. Configuração

In [1]:
import os, io
import pandas as pd
import pyodbc
import boto3
from botocore.client import Config
from dotenv import load_dotenv

load_dotenv(override=True)

DB_SERVER   = os.getenv('DB_SERVER')
DB_PORT     = os.getenv('DB_PORT')
DB_USER     = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_DATABASE = os.getenv('DB_DATABASE')

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
LANDING_BUCKET   = os.getenv('MINIO_LANDING_BUCKET')

print(f'SQL Server: {DB_SERVER}:{DB_PORT}/{DB_DATABASE}')
print(f'MinIO: {MINIO_ENDPOINT} | Bucket: {LANDING_BUCKET}')

SQL Server: localhost:1433/InsuranceDB
MinIO: http://localhost:9020 | Bucket: landing-zone


## 2. Conexão com SQL Server

In [2]:
conn = pyodbc.connect(
    f'DRIVER={{ODBC Driver 18 for SQL Server}};'
    f'SERVER={DB_SERVER},{DB_PORT};'
    f'DATABASE={DB_DATABASE};'
    f'UID={DB_USER};PWD={DB_PASSWORD};'
    f'TrustServerCertificate=yes;'
)
cursor = conn.cursor()
print(f'Conectado ao SQL Server [{DB_DATABASE}]')

Conectado ao SQL Server [InsuranceDB]


## 3. Listar Tabelas

In [3]:
cursor.execute("""
    SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_TYPE = 'BASE TABLE' AND TABLE_SCHEMA = 'dbo'
    ORDER BY TABLE_NAME
""")
tabelas = [row[0] for row in cursor.fetchall()]

print(f'{len(tabelas)} tabelas encontradas:')
for i, t in enumerate(tabelas, 1):
    cursor.execute(f'SELECT COUNT(*) FROM [{t}]')
    print(f'  {i:2d}. {t:<20} ({cursor.fetchone()[0]:>6} registros)')

7 tabelas encontradas:
   1. Agent_Policies       (    10 registros)
   2. Agents               (    10 registros)
   3. Claims               (    10 registros)
   4. Customers            (    10 registros)
   5. Insurance_Types      (    10 registros)
   6. Payments             (    10 registros)
   7. Policies             (    10 registros)


## 4. Criar Bucket no MinIO

In [4]:
s3_client = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

try:
    s3_client.head_bucket(Bucket=LANDING_BUCKET)
    print(f'Bucket [{LANDING_BUCKET}] já existe')
except:
    s3_client.create_bucket(Bucket=LANDING_BUCKET)
    print(f'Bucket [{LANDING_BUCKET}] criado!')

print('Buckets:', [b['Name'] for b in s3_client.list_buckets()['Buckets']])

Bucket [landing-zone] criado!
Buckets: ['landing-zone']


## 5. Extrair Tabelas → CSV no MinIO

In [5]:
print(f'Extraindo {len(tabelas)} tabelas...\n')
resultados = []

for tabela in tabelas:
    # Ler tabela via cursor (evita warning do pd.read_sql com pyodbc)
    cursor.execute(f'SELECT * FROM [{tabela}]')
    columns = [desc[0] for desc in cursor.description]
    rows = cursor.fetchall()
    df = pd.DataFrame.from_records(rows, columns=columns)

    csv_buffer = io.StringIO()
    df.to_csv(csv_buffer, index=False)
    csv_bytes = csv_buffer.getvalue().encode('utf-8')
    s3_key = f'{tabela}.csv'

    s3_client.put_object(
        Bucket=LANDING_BUCKET, Key=s3_key,
        Body=csv_bytes, ContentType='text/csv'
    )

    size_kb = len(csv_bytes) / 1024
    resultados.append({'tabela': tabela, 'registros': len(df), 'colunas': len(df.columns), 'tamanho_kb': round(size_kb,1)})
    print(f'  {tabela}.csv -> s3://{LANDING_BUCKET}/{s3_key} ({len(df)} registros, {size_kb:.1f} KB)')

print(f'\nExtracao concluida! {len(tabelas)} tabelas exportadas.')

Extraindo 7 tabelas...

  Agent_Policies.csv -> s3://landing-zone/Agent_Policies.csv (10 registros, 0.3 KB)
  Agents.csv -> s3://landing-zone/Agents.csv (10 registros, 0.4 KB)
  Claims.csv -> s3://landing-zone/Claims.csv (10 registros, 0.5 KB)
  Customers.csv -> s3://landing-zone/Customers.csv (10 registros, 0.7 KB)
  Insurance_Types.csv -> s3://landing-zone/Insurance_Types.csv (10 registros, 0.3 KB)
  Payments.csv -> s3://landing-zone/Payments.csv (10 registros, 0.4 KB)
  Policies.csv -> s3://landing-zone/Policies.csv (10 registros, 0.6 KB)

Extracao concluida! 7 tabelas exportadas.


## 6. Validação

In [6]:
response = s3_client.list_objects_v2(Bucket=LANDING_BUCKET)
print(f'Arquivos no bucket [{LANDING_BUCKET}]:\n')
for obj in response.get('Contents', []):
    print(f'  {obj["Key"]:<25} {obj["Size"]/1024:>8.1f} KB')

df_resumo = pd.DataFrame(resultados)
print(f'\nTotal: {df_resumo["registros"].sum():,} registros | {df_resumo["tamanho_kb"].sum():,.1f} KB')

Arquivos no bucket [landing-zone]:

  Agent_Policies.csv             0.3 KB
  Agents.csv                     0.4 KB
  Claims.csv                     0.5 KB
  Customers.csv                  0.7 KB
  Insurance_Types.csv            0.3 KB
  Payments.csv                   0.4 KB
  Policies.csv                   0.6 KB

Total: 70 registros | 3.2 KB


In [7]:
cursor.close()
conn.close()
print('Conexao SQL Server encerrada.')

Conexao SQL Server encerrada.
